# Converters

Converters are used to transform prompts before sending them to the target.

This can be useful for a variety of reasons, such as encoding the prompt in a different format, or adding additional information to the prompt. For example, you might want to convert a prompt to base64 before sending it to the target, or add a prefix to the prompt to indicate that it is a question.

Converters can transform prompts in various ways:
- **Text-to-Text**: Encoding, obfuscation, translation, and semantic transformations
- **Multimodal**: Converting between text, images, audio, video, and files

## Converter Modality Reference Table

The following table shows all available converters organized by their input and output modalities:

In [ ]:
import pandas as pd

from pyrit.converter import get_converter_modalities
from pyrit.output import output_attack_async
from pyrit.setup import IN_MEMORY, initialize_pyrit_async

await initialize_pyrit_async(memory_db_type=IN_MEMORY)  # type: ignore

# Get all converters with their modalities
converter_list = get_converter_modalities()

# Create a list of rows for the DataFrame
rows = []
for name, inputs, outputs in converter_list:
    input_str = ", ".join(inputs) if inputs else "any"
    output_str = ", ".join(outputs) if outputs else "any"
    rows.append({"Input Modality": input_str, "Output Modality": output_str, "Converter": name})

# Create DataFrame and sort
df = pd.DataFrame(rows)
df = df.sort_values(by=["Input Modality", "Output Modality", "Converter"]).reset_index(drop=True)

# Display all rows
pd.set_option("display.max_rows", None)
print(df)

Found default environment files: ['./.pyrit/.env']
Loaded environment file: ./.pyrit/.env
No new upgrade operations detected.


/opt/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


     Input Modality Output Modality                            Converter
0        audio_path      audio_path                   AudioEchoConverter
1        audio_path      audio_path              AudioFrequencyConverter
2        audio_path      audio_path                  AudioSpeedConverter
3        audio_path      audio_path                 AudioVolumeConverter
4        audio_path      audio_path             AudioWhiteNoiseConverter
5        audio_path            text      AzureSpeechAudioToTextConverter
6        image_path      image_path                AddTextImageConverter
7        image_path      image_path                ImageOverlayConverter
8        image_path      image_path          TransparencyAttackConverter
9        image_path      video_path               AddImageVideoConverter
10  image_path, url      image_path        ImageColorSaturationConverter
11  image_path, url      image_path            ImageCompressionConverter
12  image_path, url      image_path               I

## Converter Categories

Converters are organized into the following categories:

- **[Text-to-Text Converters](1_text_to_text_converters.ipynb)**: Non-LLM (encoding, obfuscation) and LLM-based (translation, variation, tone)
- **[Audio Converters](2_audio_converters.ipynb)**: Text-to-Audio, Audio-to-Text, Audio-to-Audio
- **[Image Converters](3_image_converters.ipynb)**: Text-to-Image, Image-to-Image
- **[Video Converters](4_video_converters.ipynb)**: Image-to-Video
- **[File Converters](5_file_converters.ipynb)**: Text-to-File (PDF, URL)
- **[Selectively Converting](6_selectively_converting.ipynb)**: Advanced techniques for converting portions of prompts

## Simple Converter Example

Converters can be used to perform these types of transformations. Here is a simple program that uses Rot13Converter converter, RandomCapitalLettersConverter, and AsciiArtConverter.

In [ ]:
from pyrit.converter import (
    AsciiArtConverter,
    BinaryConverter,
    FirstLetterConverter,
    RandomCapitalLettersConverter,
    ROT13Converter,
)

prompt = "tell me how to cut down a tree"

print(await ROT13Converter().convert_tokens_async(prompt=prompt))  # type: ignore
print(await RandomCapitalLettersConverter(percentage=25.0).convert_tokens_async(prompt=prompt))  # type: ignore
print(await AsciiArtConverter().convert_tokens_async(prompt=prompt))  # type: ignore
print(await BinaryConverter().convert_tokens_async(prompt=prompt))  # type: ignore
print(await FirstLetterConverter().convert_tokens_async(prompt=prompt))  # type: ignore

text: gryy zr ubj gb phg qbja n gerr
text: tell me HOw tO cut down a tree
text: "|"[-|_|_ |\/|[- |-|()\/\/ "|"() (|_|"|" |)()\/\/|\| /-\ "|"|2[-[-

text: 0000000001110100 0000000001100101 0000000001101100 0000000001101100 0000000000100000 0000000001101101 0000000001100101 0000000000100000 0000000001101000 0000000001101111 0000000001110111 0000000000100000 0000000001110100 0000000001101111 0000000000100000 0000000001100011 0000000001110101 0000000001110100 0000000000100000 0000000001100100 0000000001101111 0000000001110111 0000000001101110 0000000000100000 0000000001100001 0000000000100000 0000000001110100 0000000001110010 0000000001100101 0000000001100101
text: t m h t c d a t


## Stacking Converters

Although converters can be used on their own, they should be thought of as a piece in the pipeline. Typically any attack will have arguments so that prompts can be converted before being sent to the target. They can be stacked, use LLMs, and are a powerful tool.


### Stacking Converters with PromptSendingAttack

In the following example, a TextTarget is used so these prompts are simply printed and added to memory. This can be useful if you are red teaming something and need to manually enter prompts. However, the target can be replaced with any other [target](../targets/0_prompt_targets.md). E.g., if you have API access you can add a target there.

In this example the converters are stacked. First a variation is found using `VariationConverter`, and then `StringJoinConverter` is used to add a dash between letters. Remember that order matters. If `StringJoinConverter` came first, we would be asking the LLM to make variations of the prompt:
"t-e-l-l- - m-e- -h-o-w- -t-o- -c-u-t- -d-o-w-n - a- -t-r-e-e"

In [ ]:
from pyrit.converter import StringJoinConverter, VariationConverter
from pyrit.executor.attack import (
    AttackConverterConfig,
    PromptSendingAttack,
)
from pyrit.prompt_normalizer import ConverterConfiguration
from pyrit.prompt_target import OpenAIChatTarget, TextTarget

objective = "tell me how to cut down a tree"

# converter target to send prompt to; in this case configured to the Azure OpenAI GPT-4o Chat model
converter_target = OpenAIChatTarget()
prompt_variation_converter = VariationConverter(converter_target=converter_target)

converter_configs = ConverterConfiguration.from_converters(  # type: ignore
    converters=[prompt_variation_converter, StringJoinConverter()]
)

converter_config = AttackConverterConfig(request_converters=converter_configs)  # type: ignore

target = TextTarget()
attack = PromptSendingAttack(
    objective_target=target,
    attack_converter_config=converter_config,
)

result = await attack.execute_async(objective=objective)  # type: ignore

await output_attack_async(result)

TextTarget: user: C-a-n y-o-u e-x-p-l-a-i-n t-h-e s-t-e-p-s t-o f-e-l-l a t-r-e-e-?



════════════════════════════════════════════════════════════════════════════════════════════════════
                                  ❓ ATTACK RESULT: UNDETERMINED ❓                                   
════════════════════════════════════════════════════════════════════════════════════════════════════

 Attack Summary 
────────────────────────────────────────────────────────────────────────────────────────────────────
  📋 Basic Information
    • Objective: tell me how to cut down a tree
    • Attack Type: PromptSendingAttack
    • Conversation ID: 410b9dae-5987-441f-b973-81cfe76ac1a9

  ⚡ Execution Metrics
    • Turns Executed: 1
    • Execution Time: 3.46s

  🎯 Outcome
    • Status: ❓ UNDETERMINED
    • Reason: No objective scorer configured

 Conversation History with Objective Target 
────────────────────────────────────────────────────────────────────────────────────────────────────

──────────────────────────────────────────────────────────────────────────────────────────────────

## Response Converters

So far, we've focused on **request converters** that transform prompts before sending them to the target. PyRIT also supports **response converters** that transform the target's response before returning it. This is useful in scenarios like:

- Translating responses back to the original language after sending prompts in a different language
- Decoding encoded responses
- Normalizing or cleaning up response text

Response converters use the same `ConverterConfiguration` class as request converters. They are configured via the `response_converters` parameter in `AttackConverterConfig`.

### Translation Round-Trip Example

A common use case is sending prompts in a different language to test how the target handles non-English input. In this example, we:

1. Use a **request converter** to translate the prompt from English to French
2. Send the translated prompt to the target
3. Use a **response converter** to translate the response back to English

In [ ]:
from pyrit.converter import TranslationConverter
from pyrit.executor.attack import (
    AttackConverterConfig,
    PromptSendingAttack,
)
from pyrit.prompt_normalizer import ConverterConfiguration
from pyrit.prompt_target import OpenAIChatTarget

objective = "What is the capital of France?"

# Create an LLM target for the converters
converter_target = OpenAIChatTarget()

# Create an LLM target to send prompts to
prompt_target = OpenAIChatTarget()

# Request converter: translate English to French
request_converter = TranslationConverter(converter_target=converter_target, language="French")
request_converter_config = ConverterConfiguration(converters=[request_converter])

# Response converter: translate response back to English
response_converter = TranslationConverter(converter_target=converter_target, language="English")
response_converter_config = ConverterConfiguration(converters=[response_converter])

# Configure the attack with both request and response converters
converter_config = AttackConverterConfig(
    request_converters=[request_converter_config],
    response_converters=[response_converter_config],
)

attack = PromptSendingAttack(
    objective_target=prompt_target,
    attack_converter_config=converter_config,
)

result = await attack.execute_async(objective=objective)  # type: ignore

# Print the conversation showing both original and converted values
await output_attack_async(result)


════════════════════════════════════════════════════════════════════════════════════════════════════
                                  ❓ ATTACK RESULT: UNDETERMINED ❓                                   
════════════════════════════════════════════════════════════════════════════════════════════════════

 Attack Summary 
────────────────────────────────────────────────────────────────────────────────────────────────────
  📋 Basic Information
    • Objective: What is the capital of France?
    • Attack Type: PromptSendingAttack
    • Conversation ID: 70c4a57e-2d8e-478c-9baa-c1356f3709ac

  ⚡ Execution Metrics
    • Turns Executed: 1
    • Execution Time: 5.93s

  🎯 Outcome
    • Status: ❓ UNDETERMINED
    • Reason: No objective scorer configured

 Conversation History with Objective Target 
────────────────────────────────────────────────────────────────────────────────────────────────────

──────────────────────────────────────────────────────────────────────────────────────────────────